In [1]:
# THIS CODE USES CUPY; FOR MORE THAN 500 GRIDPOINTS
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from PIL import Image
# from numba import jit, prange
import time 

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
from scipy.signal import find_peaks
from scipy.integrate import quad
from scipy.optimize import root_scalar
from matplotlib.animation import PillowWriter, FuncAnimation
import ipywidgets as widgets
from IPython.display import display
import zarr
import json

print("Fin")

/home/nehadesigar/pixi_env/.pixi/envs/default/lib/python3.12/site-packages/cupy/_environment.py:670: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


Fin


In [2]:
# Independent parameters (free to edit)

Na = 0.5 # Units: M 
T = 303.15 # Units: K
valence = 4
duration = 250 * 10**5 # In timesteps of dt
gridpoints = 128 # Number of points
dx = 10 # Units: nm
dt = 1.E-5 # Units: sec
rho_mean = 3E-5 # Initial mean density of nanostar A, found by spinodal (rho dense + rho dilute)/2 for the value of T used
save_interval = 10**5

grid_length = dx * gridpoints # Total length (nm)
inv_dx2 = 1.0 / (dx * dx)

# Establishes constants
K = 1.0E6 # Units: nm^5 
M = 1 # Units: (nm s)^-1
vb = 1.66 # Units: nm^3
kB = 1.314E-23 * 0.24 # Units: cal/K (1J=0.24cal)
mol = 6.02E23
dHa = -42000 # Units: cal/mol 
dS1 = 1.84 * cp.log(Na) # Units: cal/mol K
dS0 = -120 # Units: cal/mol K at 1M NaCl
floor = 1E-12 # Minimum value for arrays
num_saves = duration // save_interval + 1 # Number of saved values

Da = vb * cp.exp(-(dHa - T * (dS0 + dS1)) / (mol * kB * T))
Db = Da

type = "AAAA"
length_a = 20
B2 = 2190 # Units: nm^3

print("Fin")

Fin


In [ ]:
# Initializes array of density values
cp.random.seed(7) # Opens a random number generator instance, seed 7

rho = rho_mean * (1.0 + 0.01 * cp.random.uniform(low=-1, high=1, size=(gridpoints, gridpoints, gridpoints))) # Creates rho values around the mean with slight randomness
rho = cp.maximum(rho, 1.E-10) # Prevents negative densities

initial_mass = cp.sum(rho)

def laplacian_3d(function_array):
    """
    Computes the 3D Laplacian of a function, given an array representing that function
    """
    return ( #Uses the inbuilt roll which does allow for periodic boundary conditions
        cp.roll(function_array,  1, axis=0) +
        cp.roll(function_array, -1, axis=0) +
        cp.roll(function_array,  1, axis=1) +
        cp.roll(function_array, -1, axis=1) +
        cp.roll(function_array,  1, axis=2) +
        cp.roll(function_array, -1, axis=2) -
        6.0 * function_array) * inv_dx2

beta_mu_kernel = cp.ElementwiseKernel(
    'float64 rho, float64 lap_rho',
    'float64 output',
    f'''
    output = 2.0 * {B2} * rho
          + log(rho)
          + {valence} * log((-1.0 + sqrt(1.0 + 16.0 * rho * {Da})) / (8.0 * rho * {Da}))
          - {K} * lap_rho;
    ''',
    'beta_mu_kernel'
)

def compute_step_single(rho):

    lap_rho = laplacian_3d(rho)
    
    # Total chemical potential (with floored rho, Xa)
    beta_mu_total = beta_mu_kernel(rho, lap_rho)

    laplacian_3d_mu = laplacian_3d(beta_mu_total)


    return dt * M * laplacian_3d_mu

def save_density(index, rho_total_array, output_dir, sim_params,
                  channel_colors=None):
    
    rho_final = rho_total_array[index]
    if hasattr(rho_final, "get"):
        rho_final = rho_final.get()
    rho_final = rho_final.astype(np.float32)

    if channel_colors is None:
        channel_colors = [(1, 0, 0)]  # single red channel

    zarr_path = os.path.join(output_dir, "final_density.zarr")
    root = zarr.open_group(zarr_path, mode="w")
    root.create_array("component_0", data=rho_final, chunks=(64, 64, 64))

    metadata = {
        "n_components": 1,
        "gridpoints": rho_final.shape[0],
        "channels": [{
            "name": "component_0",
            "color": channel_colors[0],
            "contrast_limits": [float(rho_final.min()), float(rho_final.max())],
        }],
        "sim_params": sim_params,
    }
    with open(os.path.join(output_dir, "render_metadata.json"), "w") as f:
        json.dump(metadata, f, indent=2)

    return zarr_path


# Initializes arrays for saving rho
num_saves = duration // save_interval + 1
rho_total_array = cp.zeros((num_saves, gridpoints, gridpoints, gridpoints)) # Fourth dimension added for 3D grid
rho_total_array[0] = rho
save_index = 1

# Tracks the mass over time to ensure conservation
mass_history = []
time_history = []

os.makedirs("OUTPUTS", exist_ok=True)
output_dir = os.path.join("OUTPUTS", f"3D_A_{length_a}_B2_{B2}")
os.makedirs(output_dir, exist_ok=True)
progress_file = os.path.join(output_dir, f"3D_{num_saves-1}_timesteps_progress.txt")


start_time = time.perf_counter()
for step in range(duration):

    # Iterates to find new value of rho
    rho += compute_step_single(rho)

    # Adds the new density to the array of densities + checks mass conservation every 10^6 steps
    if step % (save_interval) == 0:
        rho_total_array[save_index] = rho

        total_mass = cp.sum(rho)

        mass_history.append(total_mass)
        time_history.append(step * dt)

        rho = cp.maximum(rho, floor)

        time_elapsed = time.perf_counter() - start_time
        with open(progress_file, "w") as f:
            f.write(f"Progress: {step // save_interval} out of {duration // save_interval} "
                     f"at {time_elapsed:.1f} seconds\n")
                    
        if step % (save_interval * 10) == 0:
            save_density(
                save_index,
                rho_total_array,
                output_dir,
                sim_params={"B2": B2, "valence": valence, "K": K,
                            "type": type, "recent_step": step, "max": float(rho.max()), "min": float(rho.min())},
            )
        save_index += 1


In [ ]:
print(rho_total_array.max())
print(rho_total_array.min())
save_final_density(
    rho_total_array,
    output_dir,
    sim_params={"B2": B2, "valence": valence, "K": K,
                "type": type, "recent_step": duration, "max": float(rho.max()), "min": float(rho.min())})

In [ ]:
print(rho_total_array[-1].min(), rho_total_array[-1].max())
import zarr
z = zarr.open("OUTPUTS/3D_A_20_B2_2190/final_density.zarr", mode="r")
print(z[:].min(), z[:].max()) # If this is not somethign greater than 0 then we have a bit of a problem...